# PINN4SOH 模块 2：数据预处理与成对构造

### 数据流中的位置

```
.mat 
──▶ [模块1: 特征提取] 
──▶ CSV 
──▶ [模块2: 本模块] 
       │
       ├── 2a: 3σ 异常值剔除
       ├── 2b: SOH 转换 + 归一化
       ├── 2c: 时间步配对 (x₁x₂)⭐
       └── 2d: 多电池合并 + DataLoader 封装

──▶DataLoader 
──▶ [模块3: PINN]
  
```

---

## 复现运行与源码对齐说明

> 当前 clean 代码：`src/02_dataloader.py`  
> 原始代码：`dataloader/dataloader.py`  
> 执行规范：paper-reproduction；原始仓库只读。  
> 本手册已完成 Notebook JSON 与代码单元静态检查。涉及数据的单元需按 `DATA.md` 准备数据后，从头运行以生成本机结果。

本手册保留六层学习结构：纯净代码、逐行详解、语法表、数据流角色、物理含义与真实数据图、踩坑记录。论文与源码不一致处以源码为复现基准并单独说明。

### 背景：为什么需要这些预处理步骤？

**3σ 异常值剔除**：电池循环实验中可能出现传感器跳变、接触异常或其他离群点。当前代码对每一列应用均值 ±3 倍标准差规则；0.27% 只是在单变量正态分布假设下的理论尾部比例，多列联合筛选和非正态数据的实际删除比例可能更高。被删除点也不一定全部属于测量噪声，因此应结合原始曲线复核。

**归一化**：16 项统计特征和 `cycle index` 的量纲与数值尺度不同——例如电压、电流、时间、熵和循环编号。若直接输入网络，数值范围较大的时间或循环相关变量可能主导线性组合。公开源码将全部 17 个输入列归一化，仅保留 SOH 的物理比例。

**时间步配对**：电池退化不是孤立快照，是连续过程。配对让模型看到"从 t 到 t+1 发生了什么"，这是理解 L_PDE 和 L_mono 的前提。没有配对，就无法定义退化速率。

**DataLoader 封装**：按电池划分 train/test，用电池 A 训练、电池 B 测试，防止同一电池内的时序信息泄露。

---

### 模块 2 核心概念概览

> 行号记录自手册编写版本，后续整理可能产生偏移；核对实现时请以函数名和当前 `src/` 文件为准。

| 子模块 | clean_code 行号 | 原始代码行号 | 核心变换 |
|--------|:-----------:|:----------:|----------|
| 2a: 3σ 清洗 | L14-L34 | L17-41 | DataFrame → 删异常行 → DataFrame |
| 2b: SOH + 归一化 | L37-L57 | L43-66 | 容量(Ah) → SOH(0~1)，16项统计特征 + cycle index → [-1,1] |
| 2c: 时间步配对 | L60-L68 | L68-82 | N行电池数据 → N-1对 (x₁,x₂) |
| 2d: DataLoader | L71-L113 | L84-175 | 多电池数组 → TensorDataset → DataLoader |

### 目录

| 章节 | clean_code 行号 | 原始代码行号 | 内容 |
|------|:----------:|:----------:|------|
| [1. 2a: 3σ 异常值剔除](#sec1) | 14-34 | 17-41 | `_3_sigma()` + `delete_3_sigma()` |
| ↳ [1.1 代码块](#sec1) | | | 可运行代码 |
| ↳ [1.1.1 🔍 逐行详解](#sec1) | | | 每行结构角色标注（函数定义/参数/返回值…） |
| ↳ [1.2 语法速查表](#sec1) | | | 关键语法点表格 |
| [2. 2b: SOH 转换 + 归一化](#sec2) | 37-57 | 43-66 | `process_one_csv()` |
| ↳ [2.1 代码块](#sec2) | | | 可运行代码 |
| ↳ [2.1.1 🔍 逐行详解](#sec2) | | | 每行结构角色标注 + 设计决策分析 |
| ↳ [2.2 语法速查表](#sec2) | | | 关键语法点表格 |
| [3. 2c: 时间步配对 ⭐](#sec3) | 60-68 | 68-82 | `make_time_step_pairs()` |
| ↳ [3.1 代码块](#sec3) | | | 可运行代码 |
| ↳ [3.1.1 🔍 逐行详解](#sec3) | | | 每行结构角色标注 + 配对可视化 |
| ↳ [3.2 语法速查表](#sec3) | | | 关键语法点表格 |
| [4. 2d: 多电池合并 + DataLoader](#sec4) | 71-113 | 84-175 | `build_dataloaders()` |
| ↳ [4.1 代码块](#sec4) | | | 可运行代码 |
| ↳ [4.1.1 🔍 逐行详解](#sec4) | | | 四阶段拆解 + 形状追踪表 |
| ↳ [4.2 语法速查表](#sec4) | | | 关键语法点表格 |
| [5. 主程序演示](#sec5) | 116-153 | — | 完整数据流跑通 |
| ↳ [5.1 代码块](#sec5) | | | 可运行代码 |
| ↳ [5.1.1 🔍 逐行详解](#sec5) | | | 每行结构角色标注 + 主程序调用链 |

---

## 1. 2a: 3σ 异常值剔除 <a id="sec1"></a>

> **对应 src/02_dataloader.py 第 14-34 行**
> **对应原始代码 dataloader/dataloader.py 第 17-41 行**

一句话：对 DataFrame 每一列，删除超过均值 ±3 倍标准差的数据行，防止极端值污染后续归一化和训练。

### 1.1 代码块

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import os

def _3_sigma(series):
    """找出单列中超出均值±3倍标准差的数据点索引"""
    rule = (series.mean() - 3 * series.std() > series) | \
           (series.mean() + 3 * series.std() < series)
    return np.arange(series.shape[0])[rule]


def delete_3_sigma(df):
    """对所有列依次做 3σ 检测，删除任一列异常的行"""
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna()
    df = df.reset_index(drop=True)

    out_index = []
    for col in df.columns:
        index = _3_sigma(df[col])
        out_index.extend(index)
    out_index = list(set(out_index))
    df = df.drop(out_index, axis=0)
    df = df.reset_index(drop=True)
    return df

### 1.1.1  逐行详解：代码结构与每行功能

下面把代码逐行拆开，标注每一行的**结构角色**（是函数定义？是变量赋值？是循环？是返回值？）以及**具体功能**。理解代码结构是理解逻辑的前提。

---

**【模块 A】导入依赖库**

```python
import pandas as pd               # [导入模块] pandas → 数据处理框架
                                   #   核心数据结构 DataFrame（表格）和 Series（单列）
import numpy as np                # [导入模块] numpy → 数值计算库
                                   #   提供多维数组 ndarray 和数学函数（mean, std 等）
import torch                      # [导入模块] PyTorch → 深度学习框架
from torch.utils.data import TensorDataset, DataLoader
#         └──────┬──────┘    └────┬──────┘  └────┬────┘
#           torch的      [从模块导入特定类]   [从模块导入特定类]
#        数据工具子包     把多个Tensor         批量加载+打乱
#                        打包成一个数据集       +并行读取
import os                         # [导入模块] 操作系统接口 → 文件路径拼接、文件列表
```

> **关键区分**：`import X` 导入整个模块，使用时需 `X.func()`；`from X import Y` 只导入 Y，可直接用 `Y()`。

---

**【模块 B】_3_sigma 函数 —— 单列异常值检测器**

```python
def _3_sigma(series):                          # [函数定义头]
    #  └──┬──┘ └─┬──┘                           def: Python 定义函数的关键字
    #   函数名   形参                             _3_sigma: 函数名（_开头=约定为"内部函数"）
    #                                            series: 形参，接收 pandas Series（DataFrame 一列）
```

```python
    """找出单列中超出均值±3倍标准差的数据点索引"""  # [文档字符串 docstring]
    #                                              三引号包裹，描述函数功能
    #                                              可被 help() 查看，不是普通注释
```

```python
    rule = (series.mean() - 3 * series.std() > series) |  \
           (series.mean() + 3 * series.std() < series)
    # ←─ [变量赋值] ────────────────────────────────── ──  ──
    #                                                 ↑    ↑
    #                                            [按位或] [续行符]
    #
    #  逐段拆解：
    #  ┌─────────────────────────────────────────────────┐
    #  │ series.mean()            → 计算该列的均值 μ（标量）  │
    #  │ series.std()             → 计算该列的标准差 σ（标量） │
    #  │ 3 * series.std()         → 3σ，即 3 倍标准差       │
    #  │ series.mean() - 3*series.std() → 下边界 μ-3σ       │
    #  │ ... > series             → 比较：哪些数据点 < μ-3σ  │
    #  │                            返回 boolean Series     │
    #  │                            True=左尾异常（过低）     │
    #  │ series.mean() + 3*series.std() → 上边界 μ+3σ       │
    #  │ ... < series             → 比较：哪些数据点 > μ+3σ  │
    #  │                            True=右尾异常（过高）     │
    #  │ | (按位或)               → 逐元素 OR 合并左右尾      │
    #  │                            True=至少一侧异常        │
    #  │ \ (反斜杠)               → 续行符，一行写不下时用     │
    #  └─────────────────────────────────────────────────┘
    #
    #  rule 类型：pandas Series，dtype=bool
    #  rule 含义：True 的位置 = 该数据点异常（超过 3σ 范围）
```

```python
    return np.arange(series.shape[0])[rule]    # [返回语句 return]
    #       └──────────┬──────────┘ └─┬─┘
    #           [生成行号数组]       [布尔索引过滤]
    #
    #  逐段拆解：
    #  series.shape        → 返回元组，如 (1000,) 表示 1000 行
    #  series.shape[0]     → 取出第一个元素 = 行数 N
    #  np.arange(N)        → 生成 [0, 1, 2, ..., N-1] 的整数数组
    #  [rule]              → 布尔索引：只保留 rule=True 的那些行号
    #                        例如 rule = [F,T,F,T,F] → 返回 [1, 3]
    #
    #  返回类型：numpy.ndarray，dtype=int64
    #  返回内容：异常数据点所在的行索引号
```

> **函数整体角色**：`_3_sigma` 是一个**单一职责**函数——只负责"找出某一列中哪些行是异常的"。它不修改数据、不删除数据、不知道 DataFrame 有多少列。删除操作交给下面的 `delete_3_sigma`。

---

**【模块 C】delete_3_sigma 函数 —— 多列协调清洗器**

```python
def delete_3_sigma(df):                        # [函数定义头]
    #  └──────┬──────┘ └┘                        def: 定义函数关键字
    #    函数名(delete_ 前缀                    df: 形参，接收 pandas DataFrame
    #    暗示"会修改/删除数据")                     函数对 df 做清洗后返回新的 df
```

```python
    df = df.replace([np.inf, -np.inf], np.nan)   # [数据清洗 ①]
    #    └──┬──┘ └────────────┬──────────┘ └─┬─┘
    #     原df    [替换列表] 正无穷+负无穷    [替换目标] NaN
    #
    #  为什么需要这一步？
    #  • 计算中可能出现除以 0 → 得到 inf
    #  • inf 无法参与 mean()/std() 计算（结果也是 inf/NaN）
    #  • 但 dropna() 不删除 inf（inf ≠ NaN）
    #  • 所以先把 inf 转成 NaN → 下一步统一删除
```

```python
    df = df.dropna()                               # [数据清洗 ②]
    #    删除所有包含 NaN 的行（axis=0 默认按行删）
    #    删除对象：原始 NaN + 刚转换来的 NaN（原 inf）
```

```python
    df = df.reset_index(drop=True)                 # [索引重置 ①]
    #                    └────┬────┘                重置行号
    #               drop=True: 不保留旧索引为新列
    #
    #  为什么需要？
    #  dropna() 后索引会有空洞（如 0,1,3,5,7...）
    #  后续 iloc[行号] 取值会出错
    #  重置为 0,1,2,3... 连续索引
```

```python
    out_index = []                                 # [变量初始化]
    #  空列表，用来收集所有异常行的行号
    #  为什么用列表？因为后面要用 extend() 追加元素
```

```python
    for col in df.columns:                         # [循环语句 for]
    #   遍历 df 的每一列（列名）
    #   df.columns → 返回所有列名的 Index 对象
```

```python
        index = _3_sigma(df[col])                  # [函数调用] ← 调用上面定义的 _3_sigma
        #             └──┬──┘                        df[col] 取出一列 → 传给 _3_sigma
        #          取 DataFrame 的单列                 返回该列的异常行号数组
        #          结果是 Series
        #
        #  关键：每列独立计算自己的 μ±3σ 边界
        #  电压列的 3σ 范围和电流列的 3σ 范围是不同的
```

```python
        out_index.extend(index)                    # [列表操作]
        #   extend(): 把数组/列表中的每个元素追加到列表末尾
        #   效果：out_index = [3, 7, 3, 12, 7, ...]  
        #   注意：不同列可能标记同一行为异常 → 行号可能重复！
```

```python
    out_index = list(set(out_index))               # [去重操作]
    #            └──────┬──────┘
    #        set() → 集合（自动去重）→ list() 转回列表
    #        例如：[3,5,3,5,7] → {3,5,7} → [3,5,7]
```

```python
    df = df.drop(out_index, axis=0)                # [删除行]
    #    └─┬─┘ └───┬───┘ └─┬─┘
    #     原df  要删的行号   axis=0=按行删
    #    一次删除所有异常行（比逐行删除高效得多）
```

```python
    df = df.reset_index(drop=True)                 # [索引重置 ②]
    #    删除行后必须再次重置索引
    #    否则索引不连续 → 后续配对时 x[:-1] 可能取到错误行
```

```python
    return df                                      # [返回语句]
    #    返回清洗后的 DataFrame
    #    列数不变（仍 17 列），行数 ≤ 原来（删除了异常行）
```

> **函数整体角色**：`delete_3_sigma` 是一个**协调器（coordinator）**——它遍历每一列、调用 `_3_sigma` 做检测、收集所有异常行号、统一删除。这是典型的**分治模式**：检测逻辑（`_3_sigma`）和删除逻辑（`delete_3_sigma`）被分离到两个函数中，各司其职。

---

**【设计模式总结】**：注意这两个函数的分工——

| | `_3_sigma(series)` | `delete_3_sigma(df)` |
|---|---|---|
| **输入** | 一列数据（Series） | 整个表格（DataFrame） |
| **做什么** | 找出这一列的异常行号 | 遍历所有列，收集异常行号并删除 |
| **输出** | 异常行号数组 | 清洗后的 DataFrame |
| **修改数据？** | ❌ 不修改 | ✅ 删除行 |
| **职责** | 检测 | 协调 + 执行 |

### 1.2 语法逐行拆解

| 语法点 | 含义 | 这里为什么用 |
|--------|------|-------------|
| `series.mean() ± 3*series.std()` | 正态分布 3σ 边界，覆盖 99.73% 数据 | 电池特征近似正态；3σ 只删最极端 0.27%，不会误删真实老化加速信号 |
| `\|` (按位或) | 对两个 boolean Series 逐元素 OR | 同时检测左尾（过低）和右尾（过高）的异常 |
| `np.arange(series.shape[0])[rule]` | 生成行号数组，用布尔数组索引取异常行号 | `np.arange` 生成 0,1,2,... 然后 `[rule]` 保留 rule=True 的那些 |
| `df.replace([np.inf, -np.inf], np.nan)` | inf → NaN | 后续 `dropna()` 可以统一处理；inf 无法参与统计计算 |
| `reset_index(drop=True)` | 行号重置为 0,1,2... | 删除行后索引不连续，后续 `iloc` 会出错。`drop=True` 不保留旧索引列 |
| `list(set(out_index))` | 去重 | 同一行可能在多列都异常，只需删除一次 |

### 1.3 逻辑：这一步在整体流程中的角色

```
CSV 读入 (17列 × N行)
    │
    ▼  delete_3_sigma()
    ├── 遍历每一列
    ├── 对每列计算 μ ± 3σ 边界
    ├── 收集所有超出边界的行号
    ├── 去重 → 一次性删除
    └── 重置行号
    │
    ▼
清洗后 DataFrame (17列 × M行, M ≤ N)
    │
    ├── 如果跳过此步 → 归一化时一个极端值就能让 min/max 跑偏
    └── 下游：归一化
```

每一个被删的行，是因为它在**至少一列**上超出了该列的 3σ 范围。

### 1.4 物理含义

电池循环实验的异常来源：
- **传感器跳变**：接触瞬间断开 → 电压/电流读数突变
- **数据记录错误**：时间戳错位、NaN 写入
- **环境干扰**：电磁干扰导致采样值异常

这些不是电池老化产生的——老化是缓慢的、累积的过程，不会在一个循环内跳变。3σ 剔除的是测量噪声，不是老化信号。

**用真实数据验证：** 看一个 CSV 在 3σ 清洗前后行数的变化。

In [ ]:
import os, sys
ROOT = r'../data/XJTU data'

csv_path = os.path.join(ROOT, '2C_battery-1.csv')
df_raw = pd.read_csv(csv_path)
print(f'原始行数: {df_raw.shape[0]}')

df_raw.insert(df_raw.shape[1]-1, 'cycle index', np.arange(df_raw.shape[0], dtype=np.float64))

df_clean = delete_3_sigma(df_raw)
print(f'清洗后行数: {df_clean.shape[0]}')
print(f'删除行数: {df_raw.shape[0] - df_clean.shape[0]}')

print(f'\n各列 3σ 边界（前 4 列）:')
for col in df_raw.columns[:4]:
    mu, sig = df_raw[col].mean(), df_raw[col].std()
    print(f'  {col}: μ={mu:.4f}, σ={sig:.4f}, 范围=[{mu-3*sig:.4f}, {mu+3*sig:.4f}]')

### 1.5 ⚠️ 踩坑记录

| 错误做法 | 现象 | 正确做法 |
|----------|------|----------|
| 删行后不 `reset_index` | 后续 `iloc` 取到错的行，`KeyError` 或静默取错数据 | `drop` 后立即 `reset_index(drop=True)` |
| 只对 capacity 列做 3σ | 特征列的异常值照样污染归一化，导致整列缩放失效 | 对所有列遍历检测 |
| 用 `dropna()` 但不先 `replace(inf, nan)` | inf 不会被 `dropna` 删除，继续污染统计计算 | 先 replace 再 dropna |

---

## 2. 2b: SOH 转换 + 特征归一化 <a id="sec2"></a>

> **对应 src/02_dataloader.py 第 37-57 行**
> **对应原始代码 dataloader/dataloader.py 第 43-66 行**

一句话：把放电容量 (Ah) 转为 SOH（占比），把 16 个特征统一缩放到 [-1, 1]。

### 2.1 代码块

In [ ]:
def process_one_csv(file_name, nominal_capacity=None, norm_method='min-max'):
    """读取一个 CSV，返回清洗和归一化后的 DataFrame。"""
    df = pd.read_csv(file_name)
    # 使用 float64 兼容 pandas 3；cycle index 仍按公开源码参与归一化。
    df.insert(
        df.shape[1] - 1, 'cycle index',
        np.arange(df.shape[0], dtype=np.float64),
    )
    df = delete_3_sigma(df)

    if nominal_capacity is not None:
        df['capacity'] = df['capacity'] / nominal_capacity
        input_cols = [c for c in df.columns if c != 'capacity']
        f_df = df[input_cols]

        if norm_method == 'min-max':
            scale = f_df.max() - f_df.min()
            safe_scale = scale.mask(scale == 0, 1.0)
            f_df = 2 * (f_df - f_df.min()) / safe_scale - 1
            f_df.loc[:, scale == 0] = 0.0
        elif norm_method == 'z-score':
            scale = f_df.std()
            safe_scale = scale.mask(scale == 0, 1.0)
            f_df = (f_df - f_df.mean()) / safe_scale
            f_df.loc[:, scale == 0] = 0.0

        df[input_cols] = f_df.to_numpy()

    return df

### 2.1.1 🔍 逐行详解：process_one_csv 函数

这是模块 2 的**主控函数**——它把 2a（3σ 清洗）和 2b（SOH + 归一化）串联在一个函数里。一行行拆开看：

---


```python
def process_one_csv(file_name, nominal_capacity=None, norm_method='min-max'):
#   └──────┬──────┘ └───┬───┘ └────────┬────────┘ └────┬────┘
#     [函数定义头]    [形参1]     [形参2=默认值None]  [形参3=默认值'min-max']
#
#   逐参数说明：
#   • file_name: str → CSV 文件的路径
#   • nominal_capacity: float 或 None → 电池的额定容量（Ah）
#     None 表示"不做 SOH 转换"（仅清洗，不归一化）
#   • norm_method: str → 归一化方法，支持 'min-max' 或 'z-score'
```

```python
    df = pd.read_csv(file_name)                # [文件读取]
    #   └────┬────┘                                pd.read_csv(): pandas 读取 CSV → DataFrame
    #  pandas 的读取函数                              每一行 = 一个充放电循环
    #                                               每一列 = 一个特征或标签
```

```python
    # 插入 cycle index（倒数第二列，capacity 前面）
    df.insert(df.shape[1] - 1, 'cycle index', np.arange(df.shape[0], dtype=np.float64))
    # └──┬──┘ └─────┬─────┘  └─────┬─────┘ └──────────┬──────────┘
    # [DataFrame  [列位置]      [新列名]        [新列的值]
    #  方法]
    #
    #  逐段拆解：
    #  df.shape         → 返回 (行数, 列数)，如 (370, 17)
    #  df.shape[1]      → 取列数 = 17
    #  df.shape[1] - 1  → 16 = 倒数第二列的位置
    #  'cycle index'    → 新列的名称
    #  np.arange(df.shape[0]) → [0, 1, 2, ..., N-1]，每个循环的编号
    #
    #  为什么用 insert 而不是 df['cycle index'] = ...？
    #  insert 可以控制列的位置——capacity 是最后一列，
    #  cycle_index 插在它前面，保持列顺序与论文一致
```

```python
    # 3σ 清洗
    df = delete_3_sigma(df)                    # [函数调用] ← 调用 2a 中定义的函数
    #   └──────┬──────┘                          参数是 df（DataFrame）
    #      我们上面定义的                            返回清洗后的 df
    #      清洗函数                                此行可能减少行数
```

```python
    if nominal_capacity is not None:           # [条件判断 if]
    #                                ┌─ is not None: 判断是否传了额定容量
    #                                │  如果没传（None）→ 跳过分块 → 直接返回清洗后的 df
    #                                │  如果传了（如 2.0）→ 进入 SOH 转换 + 归一化
```

```python
        # 容量 → SOH（占额定容量的比例）
        df['capacity'] = df['capacity'] / nominal_capacity
        #                  └─────┬─────┘   └──────┬──────┘
        #               [取 capacity 列]    [除以额定容量（标量）]
        #
        #  Pandas 自动做逐元素除法（broadcasting）：
        #  2.0Ah / 2.0 = 1.0  (SOH=100%)
        #  1.9Ah / 2.0 = 0.95 (SOH=95%)
        #  1.6Ah / 2.0 = 0.80 (SOH=80%)
        #  结果 ∈ [0, 1]，物理含义是"剩余健康度百分比"
```

```python
        # 公开源码归一化全部输入列，仅排除 capacity
        input_cols = [c for c in df.columns if c != 'capacity']
        #             └──────────────────┬──────────────────┘
        #       等价于上游 df.iloc[:, :-1]：16项统计特征 + cycle_index
        #
        #  这行等价于：
        #  input_cols = []
        #  for c in df.columns:
        #      if c != 'capacity':
        #          input_cols.append(c)
        #
        #  结果：input_cols = 16项统计特征 + cycle_index，共17列
        #  为什么包含 cycle_index？公开源码将其作为时间输入，并和16项统计特征一起归一化
        #  为什么排除 capacity？它是标签（SOH），只做除法转换，不做归一化
```

```python
        f_df = df[input_cols]                # [列索引取子集]
        #      └──────┬──────┘                    从 df 中取出全部17个输入列
        #     用列名列表做索引                       返回一个新的 DataFrame（f_df）
        #     而不是 .iloc                           包含16项统计特征和 cycle_index，不包含 capacity
```

```python
        if norm_method == 'min-max':           # [条件分支 if-elif]
        #   判断归一化方法
```

```python
            f_df = 2 * (f_df - f_df.min()) / (f_df.max() - f_df.min()) - 1
            #      └──────────────────────┬──────────────────────┘
            #                [min-max 归一化公式]
            #
            #   逐段拆解：
            #   f_df.min()             → 每列的最小值（返回 Series，每列一个值）
            #   f_df - f_df.min()      → 平移：使最小值为 0
            #   f_df.max() - f_df.min()→ 每列的极差（动态范围）
            #   (...) / (...)          → 缩放：使范围为 [0, 1]
            #   2 * (...)              → 拉伸：使范围为 [0, 2]
            #   2 * (...) - 1          → 平移：使范围为 [-1, 1]
            #
            #   为什么最终要 [-1, 1] 而不是 [0, 1]？
            #   tanh 激活函数的输出范围是 [-1, 1]
            #   输入也在 [-1, 1] 可以避免梯度饱和
```

```python
        elif norm_method == 'z-score':         # [备选分支]
            f_df = (f_df - f_df.mean()) / f_df.std()
            #      └────────────┬────────────┘
            #           [z-score 标准化公式]
            #
            #   逐段拆解：
            #   f_df.mean() → 每列的均值 μ
            #   f_df - μ    → 中心化（均值为 0）
            #   f_df.std()  → 每列的标准差 σ
            #   ... / σ     → 缩放（标准差为 1）
            #   结果：均值为 0，标准差为 1 的标准正态分布
```

```python
        df[input_cols] = f_df.to_numpy()        # [列赋值]
        #                   └────┬────┘
        #              取 NumPy 数组（去掉列名索引）
        #
        #  cycle_index 创建时使用 float64，因此 pandas 3 可以安全写回归一化浮点值。
        #  to_numpy() 仅去掉列名索引，列集合仍与上游 df.iloc[:, :-1] 等价
```

```python
    return df                                  # [返回语句]
    #   返回处理完的 DataFrame（17列 × M行）
    #   包含：16项归一化统计特征 + 归一化 cycle_index + SOH(0~1)
```

> **函数整体角色**：`process_one_csv` 是单文件处理流水线——读入 → 清洗 → SOH 转换 → 归一化 → 输出。上层函数 `build_dataloaders` 会对每个 CSV 调用它。

---

**【关键设计决策】**：

| 决策 | 为什么这样设计 |
|------|---------------|
| `nominal_capacity=None` 作为默认值 | 允许"只清洗不归一化"的使用场景（调试时有用） |
| 归一化全部输入列 | 与公开源码一致：16项统计特征和 `cycle_index` 一起归一化，仅排除 `capacity` |
| min-max 作为默认方法 | 电池特征有明确物理边界，min-max 保留了"在边界中位置"的语义 |
| `cycle_index` 使用 float64 | 兼容 pandas 3 的 dtype 检查，同时保持上游归一化行为 |

### 2.2 语法逐行拆解

| 语法点 | 含义 | 这里为什么用 |
|--------|------|-------------|
| `df.insert(pos, name, values)` | 在指定位置插入一列 | 把 cycle_index 放在特征和 capacity 之间，保持列顺序与论文一致 |
| `df.shape[1] - 1` | 倒数第二列的位置 | capacity 是最后一列，cycle_index 插在它前面 |
| `df['capacity'] / nominal_capacity` | 逐元素除法 | 2.0Ah 额定容量 → SOH ∈ [0.8, 1.0] |
| `2*(x-min)/(max-min) - 1` | min-max 缩放到 [-1,1] | 电池特征有明确物理边界（电压 4.0~4.2V，电流 0.1~0.5A），min-max 保留"边界"语义 |
| `[c for c in df.columns if c != 'capacity']` | 列表推导式过滤列名 | 归一化16项统计特征和 cycle_index，仅排除 capacity（标签） |
| `np.arange(..., dtype=np.float64)` | 浮点型 cycle_index | 归一化结果可在 pandas 3 中安全写回 |

### 2.3 逻辑：这一步在整体流程中的角色

```
DataFrame (17列, 含原始物理量)
    │
    ├── capacity / 2.0  →  SOH ∈ [0.8, 1.0]    ← 物理意义：健康度百分比
    │
    ├── 16 特征列 min-max → [-1, 1]            ← 消除量纲差异
    │
    ▼
归一化后 DataFrame (17列, 无量纲)
    │
    └── 下游：时间步配对（2c）
```

**为什么用 min-max 而非 z-score？**

| 方法 | 公式 | 前提 | 适用 |
|------|------|------|------|
| min-max | `2(x-min)/(max-min)-1` | 数据有界 | ✅ 电池：电压/电流有明确的物理窗口 |
| z-score | `(x-μ)/σ` | 数据正态 | 无界数据 |

电池特征来自固定的 [4.0, 4.2]V 和 [0.1, 0.5]A 窗口，物理上就有上下界。min-max 保留了"这个值在窗口中处于什么位置"的语义。

### 2.4 物理含义

归一化前后的直观理解：

In [ ]:
csv_path = os.path.join(ROOT, '2C_battery-1.csv')
df = pd.read_csv(csv_path)
df.insert(df.shape[1]-1, 'cycle index', np.arange(df.shape[0], dtype=np.float64))
df = delete_3_sigma(df)

print('归一化前：')
print(f'  voltage mean 范围: [{df["voltage mean"].min():.3f}, {df["voltage mean"].max():.3f}] V')
print(f'  CC charge time 范围: [{df["CC charge time"].min():.1f}, {df["CC charge time"].max():.1f}] s')
print(f'  voltage entropy 范围: [{df["voltage entropy"].min():.1f}, {df["voltage entropy"].max():.1f}] (无量纲)')
print(f'  capacity (SOH) 范围: [{df["capacity"].min()/2.0:.4f}, {df["capacity"].max()/2.0:.4f}]')

df['capacity'] = df['capacity'] / 2.0
input_cols = [c for c in df.columns if c != 'capacity']
f_df = df[input_cols]
f_df = 2 * (f_df - f_df.min()) / (f_df.max() - f_df.min()) - 1
df[input_cols] = f_df.to_numpy()

print(f'\n归一化后（min-max → [-1,1]）：')
print(f'  voltage mean 范围: [{df["voltage mean"].min():.3f}, {df["voltage mean"].max():.3f}]')
print(f'  CC charge time 范围: [{df["CC charge time"].min():.3f}, {df["CC charge time"].max():.3f}]')
print(f'  voltage entropy 范围: [{df["voltage entropy"].min():.3f}, {df["voltage entropy"].max():.3f}]')
print(f'  SOH 范围: [{df["capacity"].min():.4f}, {df["capacity"].max():.4f}]')
print(f'\n→ 所有特征现在在 [-1,1]，神经网络不会被量纲误导')

### 2.5 ⚠️ 踩坑记录

| 错误做法 | 现象 | 正确做法 |
|----------|------|----------|
| 用整数 dtype 创建 cycle_index 后写回归一化浮点值（pandas 3） | `TypeError: Invalid value ... for dtype 'int64'` | 以 `np.float64` 创建 cycle_index，再归一化全部输入列 |
| 对 capacity 也做 min-max | SOH 被缩放到 [-1,1]，失去"百分比"的物理意义。1.0→SOH=100% 就变成了某个随机数 | capacity 只除以额定容量得到 SOH，不做 min-max |

---

## 3. 2c: 时间步配对 ⭐ <a id="sec3"></a>

> **对应 src/02_dataloader.py 第 60-68 行**
> **对应原始代码 dataloader/dataloader.py 第 68-82 行**

一句话：把 N 行的电池数据错位切成 N-1 对 (x₁,x₂)，让模型看到"从循环 i 到循环 i+1 发生了什么变化"。**这是理解 L_PDE 和 L_mono 的前提。**

### 3.1 代码块

In [ ]:
def make_time_step_pairs(df):
    """将一个电池的数据拆成相邻时间步配对"""
    x = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values

    x1, x2 = x[:-1], x[1:]
    y1, y2 = y[:-1], y[1:]

    return (x1, y1), (x2, y2)

### 3.1.1 🔍 逐行详解：make_time_step_pairs 函数

这是整个模块中**最重要的函数**。只有 6 行代码，但它定义了 PINN 模型的核心数据格式——相邻时间步的配对 (x₁, x₂)。理解这个函数是理解后续 L_PDE 和 L_mono 损失函数的前提。

---


```python
def make_time_step_pairs(df):                  # [函数定义头]
    #  └─────────┬────────┘ └┘                    def: 定义函数
    #      函数名（见名知义：                      df: 形参，接收一个电池的 DataFrame
    #      "制作时间步配对"）                          这个 df 已经过清洗和归一化
```

```python
    """将一个电池的数据拆成相邻时间步配对"""       # [文档字符串 docstring]
    #                                          描述函数的核心操作：
    #                                          一个电池 → 多对 (x₁,x₂)
```

```python
    x = df.iloc[:, :-1].values    # [特征矩阵提取]
    #   └──────┬──────┘ └┬┘                iloc: 按整数位置索引（不是列名）
    #    [整数位置索引]  [取所有行]              :  = 取所有行
    #                                       :-1 = 取第0列到倒数第二列
    #                                       .values → 转为 NumPy 数组
    #
    #  结果形状：(M, 17)  ← M 行，17 列输入
    #  17 列 = 16 个特征 + 1 个 cycle_index
```

```python
    y = df.iloc[:, -1].values     # [标签向量提取]
    #   └──────┬──────┘ └┘                iloc: 按整数位置索引
    #    [整数位置索引]  [取所有行]              :  = 取所有行
    #                                       -1 = 只取最后一列
    #                                       .values → 转为 NumPy 一维数组
    #
    #  结果形状：(M,)  ← M 个 SOH 值
    #  最后一列 = capacity 列（已在 process_one_csv 中转为 SOH）
```

```python
    x1, x2 = x[:-1], x[1:]         # [核心配对操作] ← 这是最关键的一行
    #        └─┬─┘  └─┬─┘
    #     [序列解包]  [切片偏移]
    #
    #  逐段拆解：
    #  x[:-1] → 取第 0 行到倒数第 2 行（去掉最后一行）
    #           形状：(M-1, 17)
    #           含义：配对中的"前一个时刻"的特征
    #
    #  x[1:]  → 取第 1 行到最后一行（去掉第一行）
    #           形状：(M-1, 17)
    #           含义：配对中的"后一个时刻"的特征
    #
    #  可视化理解（M=5 时）：
    #  原始 x:  [行0] [行1] [行2] [行3] [行4]
    #  x[:-1]:  [行0] [行1] [行2] [行3]          ← 配对中的 x₁
    #  x[1:]:   [行1] [行2] [行3] [行4]          ← 配对中的 x₂
    #  配对结果: (行0,行1) (行1,行2) (行2,行3) (行3,行4)
```

```python
    y1, y2 = y[:-1], y[1:]         # [标签配对操作]
    #        └─┬─┘  └─┬─┘             与上面完全相同的错位逻辑
    #     [序列解包]  [切片偏移]        但是对 y（SOH 值）操作
    #
    #  y[:-1] → 配对中"前一个时刻"的 SOH，形状 (M-1,)
    #  y[1:]  → 配对中"后一个时刻"的 SOH，形状 (M-1,)
```

```python
    return (x1, y1), (x2, y2)                  # [返回语句]
    #       └──┬──┘  └──┬──┘
    #     第一个时刻   第二个时刻
    #     (特征,SOH)  (特征,SOH)
    #
    #  返回类型：一个包含两个元组的元组
    #  ((M-1,17), (M-1,)), ((M-1,17), (M-1,))
    #
    #  为什么返回这种嵌套结构？
    #  下游 build_dataloaders 中分别收集：
    #  X1_list.append(x1); Y1_list.append(y1)
    #  X2_list.append(x2); Y2_list.append(y2)
    #  四个列表各自拼接 → 四个大矩阵
```

> **函数整体角色**：把 N 行的电池数据变成 N-1 对 (x₁, x₂)，每对代表"从循环 i 到 i+1"的一次退化步进。

---

**【核心可视化：错位配对的本质】**

```
原始数据（M=4 个循环）：

  循环0: [特征向量 f₀, SOH=0.950] ─┐
  循环1: [特征向量 f₁, SOH=0.948] ─┼─ Pair 0: x₁=f₀, y₁=0.950 → x₂=f₁, y₂=0.948
  循环2: [特征向量 f₂, SOH=0.945] ─┼─ Pair 1: x₁=f₁, y₁=0.948 → x₂=f₂, y₂=0.945
  循环3: [特征向量 f₃, SOH=0.941] ─┘  Pair 2: x₁=f₂, y₁=0.945 → x₂=f₃, y₂=0.941

  4 个循环 → 3 对（少一对，因为最后一个循环没有"下一个时刻"）
```

**【为什么配对是不可或缺的？】**

| 损失函数 | 需要什么 | 为什么需要配对 |
|----------|---------|---------------|
| L_data | x₁ 和 x₂ 分别预测 SOH | 两个时刻各自计算 MSE |
| L_PDE | 从 x₁ 到 x₂ 的退化速率 u_t | u_t = du/dt ≈ (u₂-u₁)/(t₂-t₁)，必须有两个点才能算 |
| L_mono | SOH 单调递减约束 | 比较 u₂ 和 u₁ 的大小关系，单点无法比较 |

没有配对，L_PDE 和 L_mono 完全无法定义。这就是为什么配对是这个模型设计的**结构性前提**。

### 3.2 语法逐行拆解

| 语法点 | 含义 | 这里为什么用 |
|--------|------|-------------|
| `df.iloc[:, :-1]` | 取所有行、除最后一列外的所有列 | 17 列输入 = 16 特征 + cycle_index |
| `df.iloc[:, -1]` | 取所有行、最后一列 | capacity 列 = SOH = 预测目标 |
| `.values` | DataFrame → NumPy 数组 | 后续 `x[:-1]` 切片需要 NumPy 操作 |
| `x[:-1]` | 从第 0 行到倒数第 2 行 | 去掉最后一行（最后一行没有"下一个时刻"） |
| `x[1:]` | 从第 1 行到最后一行 | 去掉第一行（第一行没有"上一个时刻"） |
| 元组返回 `(x1,y1),(x2,y2)` | 两个独立的数据对 | 下游 `load_all_battery` 分别收集并拼接 |

### 3.3 逻辑：这一步在整体流程中的角色

```
一个电池的 DataFrame (M 行 × 18 列):
  行0: [f₀₁...f₀₁₆, cycle=0, SOH=0.950]
  行1: [f₁₁...f₁₁₆, cycle=1, SOH=0.948]
  行2: [f₂₁...f₂₁₆, cycle=2, SOH=0.945]
  ...
  行M-1: [f..., cycle=M-1, SOH=0.801]

                    │  make_time_step_pairs()
                    ▼

  Pair 0: x1=行0特征, y1=0.950  →  x2=行1特征, y2=0.948
  Pair 1: x1=行1特征, y1=0.948  →  x2=行2特征, y2=0.945
  ...
  Pair M-2: x1=行M-2, y1=0.803 → x2=行M-1, y2=0.801

  总计: M-1 对
```

**为什么每个 Pair 需要 x₁ 和 x₂ 两个输入？**

```
┌─────────────────────────────────────────┐
│           PINN.forward(x₁)               │
│  u₁ = F(x₁)     ← 预测循环 i 的 SOH     │
│  f₁ = u_t₁ - G(...)  ← PDE 残差(i)     │
├─────────────────────────────────────────┤
│           PINN.forward(x₂)               │
│  u₂ = F(x₂)     ← 预测循环 i+1 的 SOH   │
│  f₂ = u_t₂ - G(...)  ← PDE 残差(i+1)   │
├─────────────────────────────────────────┤
│  L_data = ½MSE(u₁,y₁) + ½MSE(u₂,y₂)    │  ← 需要一对真值
│  L_PDE  = ½MSE(f₁,0) + ½MSE(f₂,0)     │  ← 需要一对 PDE 残差
│  L_mono = ReLU((u₂-u₁) × (y₁-y₂))      │  ← 需要相邻时刻比较！
└─────────────────────────────────────────┘
```

**如果不配对，L_PDE 只能算一个残差、L_mono 根本无法计算。** 配对是这个模型设计的结构性前提。

### 3.4 物理含义

电池退化是一个**连续过程**，而非孤立快照。从循环 i 到循环 i+1：
- SOH 应该单调递减（y₂ ≤ y₁），除非容量回升
- 退化速率 u_t = ∂SOH/∂t 描述了"每过一个循环 SOH 掉了多少"
- 退化速率只有跨越时间才能测量——这就是为什么需要两个时间点

用真实数据看出成对关系：

In [ ]:
IMAGE_DIR = r'../outputs/figures'
import matplotlib.pyplot as plt

csv_path = os.path.join(ROOT, '2C_battery-1.csv')
df = pd.read_csv(csv_path)
df.insert(df.shape[1]-1, 'cycle index', np.arange(df.shape[0], dtype=np.float64))
df = delete_3_sigma(df)
df['capacity'] = df['capacity'] / 2.0

(x1, y1), (x2, y2) = make_time_step_pairs(df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(range(len(y1)+1), [y1[0]] + list(y2), 'b-', alpha=0.5, linewidth=0.8)
ax.scatter(range(len(y1)+1), [y1[0]] + list(y2), s=8, c='blue', alpha=0.6)
ax.set_xlabel('Cycle')
ax.set_ylabel('SOH')
ax.set_title(f'SOH 退化曲线 ({len(y1)+1} 个循环 → {len(y1)} 对)')
ax.grid(True, alpha=0.3)

ax.annotate('Pair 0', xy=(0, y1[0]), xytext=(15, y1[0]+0.005),
            arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=10)
ax.annotate('', xy=(1, y2[0]), xytext=(0, y1[0]),
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

ax = axes[1]
deltas = y2 - y1
colors = ['red' if d > 0 else 'green' for d in deltas]
ax.bar(range(len(deltas)), deltas, color=colors, alpha=0.6, width=1.5)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Pair index')
ax.set_ylabel('ΔSOH (y₂ - y₁)')
ax.set_title(f'SOH 变化量分布 (红=上升/容量回升, 绿=下降/正常退化)')
ax.grid(True, alpha=0.3)

n_up = (deltas > 0).sum()
n_down = (deltas <= 0).sum()
print(f'总配对数: {len(deltas)}')
print(f'SOH 下降 (正常退化): {n_down} 对 ({n_down/len(deltas)*100:.1f}%)')
print(f'SOH 上升 (容量回升): {n_up} 对 ({n_up/len(deltas)*100:.1f}%)')
print(f'\n→ 大部分对 SOH 下降，L_mono 惩罚的是那些上升的少数异常')

plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, 'time_step_pairs.png'), dpi=150, bbox_inches='tight')
plt.show()

### 3.5 ⚠️ 踩坑记录

| 错误做法 | 现象 | 正确做法 |
|----------|------|----------|
| 把所有电池拼接后再配对 | 电池 A 的最后一行和电池 B 的第一行组成"假对"——它们根本不是同一个电池的相邻循环 | 每个电池独立配对后再拼接 |
| 忘记 `[:-1]` 和 `[1:]` 长度差 1 | x1 和 x2 长度不对齐，torch 报 shape mismatch | 理解：M 行 → M-1 对，最后一对是 (行M-2, 行M-1) |
| 在归一化前配对 | x1 和 x2 来自不同行，如果归一化是用全局 min/max 则没问题；如果是按行归一化则会破坏配对一致性 | 先归一化再配对 |

---

## 4. 2d: 多电池合并 + DataLoader 封装 <a id="sec4"></a>

> **对应 src/02_dataloader.py 第 71-113 行**
> **对应原始代码 dataloader/dataloader.py 第 84-175 行**

一句话：把多个电池的配对数据拼接 → 按 8:2 时序划分 train/test → 从 train 中随机抽 20% 做 valid → 包装成 PyTorch DataLoader。

### 4.1 代码块

In [ ]:
import sys
from pathlib import Path
CLEAN = Path(r'../src')
sys.path.insert(0, str(CLEAN))
from module_loader import load_clean_module
data_module = load_clean_module('02_dataloader.py', 'notebook_data')
build_dataloaders = data_module.build_dataloaders
process_one_csv = data_module.process_one_csv
make_time_step_pairs = data_module.make_time_step_pairs
delete_3_sigma = data_module.delete_3_sigma
print('已加载 canonical clean 数据模块')

### 4.1.1 🔍 逐行详解：build_dataloaders 函数

这是模块 2 的**顶层入口函数**——它通过 `_load_tensors()` 完成多电池处理、时间步配对、拼接和 Tensor 转换，再构造三套划分策略，最终返回 6 个 DataLoader。下面按"A/B/C/D 四个阶段"拆解；阶段 A/B 展开说明 `_load_tensors()` 内部完成的工作。

---

**【前置：额外导入】**
```python
from sklearn.model_selection import train_test_split
#     └─────┬─────┘                   └──────┬──────┘
#   [从 sklearn 导入]                  [train_test_split 函数]
#   scikit-learn 机器学习库            功能：按比例随机划分数据集
#
from torch.utils.data import TensorDataset, DataLoader
#     └────┬────┘        └─────┬─────┘  └────┬────┘
#   [从 PyTorch 导入]    [TensorDataset 类]  [DataLoader 类]
import torch              # [导入 PyTorch] → 提供 torch.from_numpy() 等
```

---

**【阶段 A：多电池处理 + 配对收集】**

```python
def build_dataloaders(path_list, nominal_capacity, batch_size=256,
                      norm_method="min-max"):
#   path_list: CSV 路径列表；nominal_capacity: 额定容量(Ah)
#   batch_size: 每批样本数；norm_method: 特征归一化方法
```

```python
    X1_list, X2_list, Y1_list, Y2_list = [], [], [], []
    # [变量初始化] 四个空列表
    #   X1_list: 存放每个电池的 x1（前时刻特征）
    #   X2_list: 存放每个电池的 x2（后时刻特征）
    #   Y1_list: 存放每个电池的 y1（前时刻 SOH）
    #   Y2_list: 存放每个电池的 y2（后时刻 SOH）
    #   为什么用列表？因为每个电池的行数不同，不能直接堆叠
    #   先收集再最后一次性拼接
```

```python
    for path in path_list:                     # [循环语句]
    #   遍历每个 CSV 文件路径
    #   每次循环处理一节电池
```

```python
        df = process_one_csv(path, nominal_capacity=nominal_capacity)
        #   └──────┬──────┘                           ↑
        #   调用 2b 的函数（清洗+归一化）                  关键：传入额定容量
        #   返回：归一化后的 DataFrame（17列 × M行）
```

```python
        (x1, y1), (x2, y2) = make_time_step_pairs(df)
        # └──────┬──────┘   └─────────┬─────────┘
        #   [元组解包]         调用 2c 的配对函数
        #   ((M-1,17), (M-1,)), ((M-1,17), (M-1,))
```

```python
        X1_list.append(x1); X2_list.append(x2)
        Y1_list.append(y1); Y2_list.append(y2)
        # [列表追加] 把这节电池的配对数据分别放进四个列表
        #   ; 号 = 一行写多条语句（不推荐但有人用）
        #   此时：X1_list = [电池1的x1, 电池2的x1, ...]
```

---

**【阶段 B：拼接 + 转 Tensor】**

```python
    # 拼接所有电池
    X1 = np.concatenate(X1_list, axis=0)
    #    └─────┬─────┘  └───┬───┘  └─┬─┘
    #   [numpy 拼接函数]  [列表]  [axis=0=沿行方向拼接]
    #
    #   例如：电池1 (350,17) + 电池2 (360,17) → X1 (710,17)
    #   所有电池的配对数据沿行方向堆成一个大的特征矩阵
```

```python
    X2 = np.concatenate(X2_list, axis=0); Y1 = np.concatenate(Y1_list, axis=0)
    Y2 = np.concatenate(Y2_list, axis=0)
    #   同样逻辑拼接 X2, Y1, Y2
    #   Y1/Y2 是一维数组拼接 → (总配对数,) 形状
```

```python
    # 转 Tensor
    t_X1 = torch.from_numpy(X1).float()
    #      └──────┬──────┘  └─┬─┘
    #    [NumPy → PyTorch]  [转 float32]
    #
    #   NumPy 默认 float64，但 PyTorch 模型用 float32
    #   float64 → float32: 精度够用，显存减半
```

```python
    t_X2 = torch.from_numpy(X2).float()
    t_Y1 = torch.from_numpy(Y1).float().view(-1, 1)
    #                                    └───┬───┘
    #                                 [重塑张量形状]
    #   .view(-1, 1):
    #     -1 = 自动推断这一维的大小
    #      1 = 增加一维，变成列向量
    #   效果：(N,) → (N, 1)
    #   为什么需要？模型输出是 (batch, 1)，
    #   标签也是 (batch, 1) 才能对齐计算 MSE
```

```python
    t_Y2 = torch.from_numpy(Y2).float().view(-1, 1)
```

---

**【阶段 C：构造三套数据划分策略（关键！）】**

当前 canonical 实现先得到拼接后的四个长 Tensor：

```python
    x1, x2, y1, y2 = _load_tensors(path_list, nominal_capacity, norm_method)
```

随后同时构造三套用途不同的数据：

```python
    # 策略 ①：拼接后顺序切分
    split = int(x1.shape[0] * 0.8)
    first_train = (x1[:split], x2[:split], y1[:split], y2[:split])
    first_test = (x1[split:], x2[split:], y1[split:], y2[split:])
    split_values = train_test_split(*first_train, test_size=0.2, random_state=420)
    train = (split_values[0], split_values[2], split_values[4], split_values[6])
    valid = (split_values[1], split_values[3], split_values[5], split_values[7])

    # 策略 ②：全部数据全局随机切分，不预留 test
    split_values_2 = train_test_split(x1, x2, y1, y2,
                                      test_size=0.2, random_state=420)
    train_2 = (split_values_2[0], split_values_2[2],
               split_values_2[4], split_values_2[6])
    valid_2 = (split_values_2[1], split_values_2[3],
               split_values_2[5], split_values_2[7])

    # 策略 ③：不切分，完整保留传入 path_list 的全部数据
    test_3 = (x1, x2, y1, y2)
```

策略①最终约占全部数据的 64%/16%/20%，对应 `train/valid/test`；策略②对应 `train_2/valid_2`；策略③对应 `test_3`。注意：这里的顺序切分针对**多电池拼接后的全局行序列**，不等同于对每节电池分别做前后时序切分。

下面继续展开策略①的切片和验证集划分细节：

```python
    # 拼接后前 80% 作为训练池，后 20% 作为测试池
    split = int(t_X1.shape[0] * 0.8)
    #       └──┬──┘ └────┬────┘ └┬┘
    #    [取整]  [总配对数]  [比例]
    #
    #   例：总配对数 2217 → split = int(2217*0.8) = 1773
    #   前 1773 对 = train，后 444 对 = test
```

```python
    train_X1, test_X1 = t_X1[:split], t_X1[split:]
    #                    └────┬────┘  └────┬────┘
    #                 [切片：前split行] [切片：split到末尾]
    #
    #   这里保留拼接后的行顺序，不先做全局随机打乱
```

```python
    train_X2, test_X2 = t_X2[:split], t_X2[split:]
    train_Y1, test_Y1 = t_Y1[:split], t_Y1[split:]
    train_Y2, test_Y2 = t_Y2[:split], t_Y2[split:]
    #   同样对 X2, Y1, Y2 做前/后切分
```

```python
    # 训练集中随机抽 20% 做验证集
    tr_X1, v_X1, tr_X2, v_X2, tr_Y1, v_Y1, tr_Y2, v_Y2 = \
        train_test_split(train_X1, train_X2, train_Y1, train_Y2,
    #   └──────────────┬──────────────┘
    #    [sklearn 的随机划分函数]
    #    输入：四个训练张量（X1,X2,Y1,Y2 分别划分）
    #    输出：八个张量（四个 train + 四个 valid）
                         test_size=0.2, random_state=420)
    #                       └────┬────┘  └─────┬─────┘
    #                  [验证集占20%]  [随机种子=420（固定可复现）]
    #
    #  注意：策略①只在 first_train 内随机抽取验证集
```

---

**【阶段 D：封装 DataLoader】**

```python
    train_loader = DataLoader(
    #              └────┬────┘
    #           PyTorch 的数据加载器
        TensorDataset(tr_X1, tr_X2, tr_Y1, tr_Y2),
        # └─────┬──┘ └───────────────┬──────────┘
        # [TensorDataset]     四个训练张量打包成一个数据集
        # 每个样本 = (x1[i], x2[i], y1[i], y2[i]) 的四元组
        batch_size=batch_size, shuffle=True)
        #             └────┬────┘   └───┬───┘
        #          [每批样本数]    [是否打乱顺序]
        #
        #  训练集 shuffle=True：打散数据分布，防止模型记住顺序
```

```python
    valid_loader = DataLoader(
        TensorDataset(v_X1, v_X2, v_Y1, v_Y2),
        batch_size=batch_size, shuffle=True)
        #                       验证集也 shuffle=True
        #                       因为验证只算 loss，不需要保持顺序
```

```python
    test_loader = DataLoader(
        TensorDataset(test_X1, test_X2, test_Y1, test_Y2),
        batch_size=batch_size, shuffle=False)
        #                       【关键】测试集 shuffle=False
        #                       因为需要按顺序输出才能画退化曲线
        #                       乱序了就不知道哪个预测对应哪个真值
```

```python
    return {
        "train": _loader(train, batch_size, True),
        "valid": _loader(valid, batch_size, True),
        "test": _loader(first_test, batch_size, False),
        "train_2": _loader(train_2, batch_size, True),
        "valid_2": _loader(valid_2, batch_size, True),
        "test_3": _loader((x1, x2, y1, y2), batch_size, False),
    }
    # 每个 DataLoader 的 TensorDataset 都保存 (x1, x2, y1, y2)
    # train/valid/train_2/valid_2 使用 shuffle=True
    # test/test_3 使用 shuffle=False，便于保持评估输出顺序
```

> **函数整体角色**：`build_dataloaders` 是整个数据预处理管道的**入口和出口**。输入一组 CSV 路径 → `_load_tensors()` 完成清洗/归一化/配对/拼接 → 同时建立三种划分策略 → 输出 6 个 DataLoader。

---

**【完整调用链可视化】**

```
build_dataloaders(path_list, 2.0, batch_size=256)
│
├─ for path in path_list:                 ← [阶段 A] 遍历每节电池
│    ├─ process_one_csv(path, 2.0)        ← 2b: 清洗 + SOH + 归一化
│    │    └─ delete_3_sigma(df)           ← 2a: 3σ 异常值剔除
│    │         └─ _3_sigma(series) × 17   ← 每列独立检测
│    └─ make_time_step_pairs(df)          ← 2c: N行 → N-1对
│
├─ np.concatenate(...)                    ← [阶段 B] 拼接所有电池
├─ torch.from_numpy(...).float()          ← 转 Tensor (float64→float32)
│
├─ [:split] / [split:]                    ← [策略①] 顺序训练池 / 测试池
├─ train_test_split(first_train, ...)     ← [策略①] 得到 train / valid
├─ train_test_split(x1,x2,y1,y2, ...)     ← [策略②] 得到 train_2 / valid_2
├─ (x1,x2,y1,y2)                          ← [策略③] 完整 test_3
│
└─ DataLoader(TensorDataset(...)) × 6     ← [阶段 D] 封装 DataLoader
```

**【数据流形状追踪】**

| 阶段 | X1 形状 | Y1 形状 | 说明 |
|------|---------|---------|------|
| CSV 读入 | (370, 17) | (370,) | 1 节电池原始数据 |
| process_one_csv 后 | (368, 17) | (368,) | 清洗可能少几行 |
| make_time_step_pairs 后 | (367, 17) | (367,) | N→N-1 对 |
| 6 节电池 concatenate 后 | (2202, 17) | (2202,) | 拼接 |
| 转 Tensor | (2202, 17) | (2202, 1) | .view(-1,1) 加维度 |
| 80:20 时序划分 | train:(1761,17), test:(441,17) | train:(1761,1), test:(441,1) | |
| 训练→train:valid | tr:(1408,17), v:(353,17) | tr:(1408,1), v:(353,1) | 80:20 随机 |
| DataLoader (batch=256) | (256, 17) per batch | (256, 1) per batch | 最后一个 batch 可能不满 256 |

### 4.2 语法逐行拆解

| 语法点 | 含义 | 这里为什么用 |
|--------|------|-------------|
| `np.concatenate(list, axis=0)` | 沿第 0 维（行）拼接多个数组 | 6 节训练电池的配对数据拼成一个大矩阵 |
| `torch.from_numpy(arr).float()` | NumPy → PyTorch Tensor，转 float32 | PyTorch 模型只接受 Tensor；float64 → float32 节省显存 |
| `.view(-1, 1)` | 重塑张量形状：-1=自动推断, 1=1列 | y 是一维数组，需要变成 (N,1) 列向量和模型输出对齐 |
| `int(shape[0] * 0.8)` | 计算 80% 分界点 | 策略①将拼接后的前 80% 作为训练池、后 20% 作为测试池 |
| `train_test_split(..., test_size=0.2, random_state=420)` | sklearn 的随机划分 | 策略①从训练池划出验证集；策略②直接划分全部数据；固定种子保证可复现 |
| `TensorDataset(*tensors)` | 把多个 Tensor 打包成一个数据集 | 每个样本 = (x1, x2, y1, y2) 四元组 |
| `DataLoader(ds, batch_size, shuffle)` | 批量化 + 打乱 | training 需要 shuffle 打散分布；test 不需要（顺序评估） |

### 4.3 逻辑：这一步在整体流程中的角色

```
3 节电池 CSV             3 节电池 CSV
    │                        │
    ├─ 配对 (x1,x2,y1,y2)    ├─ 配对
    ▼                        ▼
  每条 ~350 对             每条 ~350 对
    │                        │
    └──── np.concatenate ────┘
              │
              ▼
         ~2100 对 (按文件及电池内部顺序拼接)
              │
    ┌─────────┴──────────┐
    ▼                    ▼
拼接后前 80%          拼接后后 20%
    │                    │
    ├─随机抽 20%         └──▶  test_loader
    ▼                       (shuffle=False)
 valid_loader
    │
    ▼
 train_loader (shuffle=True)
```

上图只表示**策略①**。由于多节电池已经拼接，前后 80/20 是全局行序切分，并不保证每节电池都按自身的早期/晚期切分。除此之外，本函数还并行返回：

| 策略 | 输出键 | 划分方式 |
|------|--------|----------|
| ① | `train/valid/test` | 拼接后前80%为训练池，训练池再随机80/20；后20%为测试 |
| ② | `train_2/valid_2` | 全部数据直接随机80/20，不预留 test |
| ③ | `test_3` | 全部数据不切分 |

**为什么 test_loader 不 shuffle？**
- 评估时需要按顺序输出，才能画退化曲线对比
- shuffle 了结果无法和真值一一对应

### 4.4 物理含义

用真实数据看划分效果：

In [ ]:
ROOT = r'../data/XJTU data'
BATCH = '2C'

names = [f for f in os.listdir(ROOT) if BATCH in f]
train_files = [os.path.join(ROOT, f) for f in names if not ('4' in f or '8' in f)]
test_files  = [os.path.join(ROOT, f) for f in names if '4' in f or '8' in f]

train_dict = build_dataloaders(train_files, nominal_capacity=2.0)
test_dict  = build_dataloaders(test_files, nominal_capacity=2.0)

def count_samples(loader):
    return sum(len(batch[0]) for batch in loader)

print(f'训练集: {len(train_files)} 节电池, {count_samples(train_dict["train_2"])} 对')
print(f'验证集: {count_samples(train_dict["valid_2"])} 对')
print(f'测试集: {len(test_files)} 节电池, {count_samples(test_dict["test_3"])} 对')

all_y1 = []
for _, _, y1, _ in train_dict['train_2']:
    all_y1.append(y1.numpy())
all_y1 = np.concatenate(all_y1)
print(f'\n训练集 SOH 范围: [{all_y1.min():.4f}, {all_y1.max():.4f}]')
print(f'→ SOH 从 {all_y1.max():.2%} 退化到 {all_y1.min():.2%}')

### 4.5 ⚠️ 踩坑记录

| 错误做法 | 现象 | 正确做法 |
|----------|------|----------|
| `batch_size` 太大（如 1024） | 小数据集一个 batch 装不下全部，训练不稳定 | 256 是经验值，小数据集用 128 |
| test_loader 设 `shuffle=True` | 评估结果和真值对不上，退化曲线乱序 | test_loader 永远 `shuffle=False` |
| 先拼接不同电池的原始行再做时间步配对 | 会把电池 A 的末行和电池 B 的首行错误配成相邻时刻 | 必须先对每节电池独立配对，再拼接各电池的 Pair |

---

## 5. 主程序演示 <a id="sec5"></a>

> **对应 src/02_dataloader.py 第 116-153 行**

完整跑通数据流：CSV → 清洗 → 归一化 → 配对 → DataLoader，展示一个 batch 的形状。

### 5.1 代码块

In [ ]:
if __name__ == '__main__':
    ROOT = r'../data/XJTU data'
    BATCH = '2C'
    NOMINAL_CAPACITY = 2.0

    all_files = [f for f in os.listdir(ROOT) if BATCH in f]

    train_files = [os.path.join(ROOT, f) for f in all_files
                   if not ('4' in f or '8' in f)]
    test_files = [os.path.join(ROOT, f) for f in all_files
                  if '4' in f or '8' in f]

    print(f'训练电池: {len(train_files)} 节')
    print(f'测试电池: {len(test_files)} 节')
    print()

    train_dict = build_dataloaders(train_files, NOMINAL_CAPACITY)
    test_dict = build_dataloaders(test_files, NOMINAL_CAPACITY)

    train_loader = train_dict['train']
    valid_loader = train_dict['valid']
    test_loader = test_dict['test']

    print(f'Train batches: {len(train_loader)}')
    print(f'Valid batches: {len(valid_loader)}')
    print(f'Test batches:  {len(test_loader)}')

    for x1, x2, y1, y2 in train_loader:
        print(f'\n一个 batch 的张量形状:')
        print(f'  x1: {list(x1.shape)}')
        print(f'  x2: {list(x2.shape)}')
        print(f'  y1: {list(y1.shape)}')
        print(f'  y2: {list(y2.shape)}')
        print(f'\n  前 3 个样本的 SOH 对 (y1 → y2):')
        for i in range(3):
            print(f'    {y1[i].item():.4f} → {y2[i].item():.4f}  '
                  f'(delta = {(y2[i]-y1[i]).item():+.6f})')
        break

### 5.1.1 🔍 逐行详解：主程序演示

这是笔记本的最终执行单元——把前面所有函数串起来，完整跑一遍数据流。每个batch 出来的是什么形状，在这里亲眼看到。

---

```python
if __name__ == '__main__':
#  └────────────┬────────────┘
#     [Python 特殊变量检查]
#     含义：只有当这个文件被直接运行时（python 02_dataloader.py），
#     才执行下面的代码。如果被 import，则跳过。
#     这是 Python 脚本的标准写法，防止 import 时误执行主程序。
```

```python
    ROOT = r'../data/XJTU data'
    #      └──────────────────────────────────┬──────────────────────────────────┘
    #         [变量赋值] 原始字符串（r'' = raw string，反斜杠不转义）
    #         XJTU 电池数据集所在目录
```

```python
    BATCH = '2C'
    #       [变量赋值] 选择 2C 倍率批次
    #       可选：'0.5C', '1C', '2C', '3C', '4C'
```

```python
    NOMINAL_CAPACITY = 2.0  # Ah
    #                  [变量赋值] 2C 批次电池的额定容量 = 2.0 Ah
    #                  用于 df['capacity'] / 2.0 → SOH
```

```python
    all_files = [f for f in os.listdir(ROOT) if BATCH in f]
    #           └┬┘ └──────────┬─────────┘ └────┬────┘
    #      [列表推导式]  [os.listdir: 列出目录下  [过滤：文件名包含 '2C']
    #                    所有文件和文件夹名]
    #
    #   等价于：
    #   all_files = []
    #   for f in os.listdir(ROOT):
    #       if '2C' in f:
    #           all_files.append(f)
    #
    #   结果：all_files = ['2C_battery-1.csv', '2C_battery-2.csv', ...]
```

```python
    # 电池 4 和 8 做测试（论文原设定）
    train_files = [os.path.join(ROOT, f) for f in all_files
    #              └─────┬─────┘              └────┬────┘
    #           [路径拼接：ROOT + 文件名]     [遍历 all_files]
                   if not ('4' in f or '8' in f)]
    #              └──────────┬──────────┘
    #          [过滤条件：排除电池4和电池8]
    #
    #  电池编号规则：文件名中有数字，如 '2C_battery-1', '2C_battery-4'
    #  排除 '4' 和 '8' → 余下 6 节电池做训练
```

```python
    test_files  = [os.path.join(ROOT, f) for f in all_files
                   if '4' in f or '8' in f]
    #              [只保留电池4和电池8 → 2 节做测试]
```

```python
    print(f'训练电池: {len(train_files)} 节')
    print(f'测试电池: {len(test_files)} 节')
    # [输出信息] f'' = f-string，大括号内是 Python 表达式
    #   输出示例：训练电池: 6 节 / 测试电池: 2 节
```

```python
    train_dict = build_dataloaders(train_files, NOMINAL_CAPACITY)
    test_dict  = build_dataloaders(test_files, NOMINAL_CAPACITY)
    #            └───────┬───────┘ └─────┬──────┘ └───────┬───────┘
    #          [调用我们上面定义的] [训练文件列表] [额定容量 2.0]
    #           顶层入口函数
    #
    #   返回：{'train': DataLoader, 'valid': DataLoader, 'test': DataLoader}
```

```python
    train_loader = train_dict['train']
    valid_loader = train_dict['valid']
    test_loader  = test_dict['test']
    # [从字典取值] 用键名取出对应的 DataLoader
```

```python
    print(f'Train batches: {len(train_loader)}')
    print(f'Valid batches: {len(valid_loader)}')
    print(f'Test batches:  {len(test_loader)}')
    # len(DataLoader) = 批次数 = ceil(样本数 / batch_size)
    # 例如：1408 个样本 / 256 = 5.5 → 6 个 batch
    #      最后一个 batch 只有 128 个样本
```

```python
    # 看一个 batch 的形状
    for x1, x2, y1, y2 in train_loader:
    #   └──────┬──────┘  └────┬────┘
    #   [元组解包：每个batch]  [迭代 DataLoader]
    #   返回 4 个 Tensor
    #   DataLoader 每次 yield 一个 batch
```

```python
        print(f'\n一个 batch 的张量形状:')
        print(f'  x1: {list(x1.shape)}   ← 循环 i 的特征 (17列: 16特征 + cycle_index)')
        print(f'  x2: {list(x2.shape)}   ← 循环 i+1 的特征')
        print(f'  y1: {list(y1.shape)}   ← 循环 i 的 SOH')
        print(f'  y2: {list(y2.shape)}   ← 循环 i+1 的 SOH')
        #   list(tensor.shape) → 如 [256, 17] → 256 个样本，每个 17 个特征
```

```python
        print(f'\n  前 3 个样本的 SOH 对 (y1 → y2):')
        for i in range(3):
        #          └─┬──┘            循环变量 i = 0, 1, 2
        #       [range(3) 生成 [0,1,2]]
```

```python
            delta = y2[i].item() - y1[i].item()
            #       └────┬────┘      └────┬────┘
            #   [.item(): Tensor → Python float]
            #   取出单个标量值（只能对只有一个元素的 Tensor 使用）
```

```python
            print(f'    {y1[i].item():.4f} → {y2[i].item():.4f}  '
            #                      └┬┘                             
            #               [:.4f = 保留4位小数]
                  f'(delta = {delta:+.6f})')
            #                      └┬┘
            #               [+: 总是显示正负号]
            #   输出示例：0.9500 → 0.9483  (delta = -0.001667)
```

```python
        break
        # [跳出循环] 只看第一个 batch 就停止
        # 否则会遍历所有 batch（太多输出）
```

> **整体角色**：这个 `if __name__ == '__main__':` 块是整个模块的**集成测试**——不需要外部调用，直接 `python 02_dataloader.py` 就能跑通整个数据流并检查输出形状。

---

**【主程序与函数的对应关系】**

```
主程序执行流程                         对应的函数
─────────────────────────────────────────────────────
ROOT / BATCH 设置                      无（配置变量）
all_files = 过滤 CSV                    无（列表推导式）
train_files / test_files 划分            无（字符串过滤）
build_dataloaders(train_files, 2.0)     → 2d 顶层函数
  └─ process_one_csv(path, 2.0)         → 2b 单文件处理
       └─ delete_3_sigma(df)            → 2a 3σ 清洗
            └─ _3_sigma(series)         → 2a 单列检测
  └─ make_time_step_pairs(df)           → 2c 时间步配对
  └─ np.concatenate / Tensor / 划分      → 2d 内部操作
  └─ DataLoader(TensorDataset(...))     → 2d 封装
print batch 形状                        无（演示输出）
```

### 5.2 完整数据流总结

```
2C CSV 文件 × 8
    │
    ├── train_files × 6：battery-1/2/3/5/6/7
    │       │
    │       │  [2a] 每节电池独立进行 3σ 清洗
    │       │  [2b] SOH 转换 + min-max 归一化
    │       │  [2c] 每节电池独立配对 → (x₁,x₂,y₁,y₂)
    │       ▼
    │   6 组配对数据分别拼接 → Tensor，共 2171 对
    │       │
    │       │  [策略②] 全局随机 80/20 切分
    │       ▼
    │   train_dict['train_2']：1736 对，batch=256，shuffle=True
    │   train_dict['valid_2']： 435 对，batch=256，shuffle=True
    │
    └── test_files × 2：battery-4/8
            │
            │  [2a/2b/2c] 每节电池独立清洗、归一化和配对
            ▼
        2 组配对数据分别拼接 → Tensor，共 732 对
            │
            │  [策略③] 全集不切分
            ▼
        test_dict['test_3']：732 对，batch=256，shuffle=False

train_2 / valid_2 / test_3 三个实际使用的 DataLoader
    │
    ▼
[模块 3: PINN.forward()]  ← 下一步从这里开始
```

---

### 与原论文的已知差异

| 差异项 | 程度 | 原因 |
|--------|------|------|
| pandas 3.0 兼容性 | 兼容性修改 | 以 float64 创建 cycle_index，再对全部输入列赋值；数学行为与上游一致 |

---